# Med3D ➜ LoCalPFN (Step-by-step, then all-at-once)

This notebook reflects the current pipeline behavior:
- Folder-level validation split is used for caching/extraction only.
- Evaluation uses a feature-level STRATIFIED split from the union of features (train+val).
- A single-method orchestrator is provided for multi-dataset runs.

We demonstrate LoCalPFN with fast defaults for quick iteration.

In [ ]:
# Environment: cap threads to mitigate OpenMP runtime conflicts / oversubscription
import os, sys, site, torch
os.environ['PYTHONNOUSERSITE'] = '1'
try:
    usr = site.getusersitepackages()
    sys.path = [p for p in sys.path if p != usr]
except Exception:
    pass
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'
try:
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Thread caps applied. Device:', device_str)

In [ ]:
# Locate repo root so imports and relative paths work regardless of launch dir
from pathlib import Path
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")
repo_root = _add_repo_root_to_sys_path()
print('Repo root:', repo_root)
config_path = repo_root / 'configs' / 'datasets.yaml'
outputs_base = repo_root / 'notebooks'
print('Config path:', config_path)
print('Outputs base:', outputs_base)

## Steps 1–2: Prepare dataset for SAM-Med3D

In [ ]:
from med3pipe.data import prepare_for_sam3d, find_default_sam3d_root
SAM3D_ROOT = find_default_sam3d_root()
CATEGORY = 'gist'
CT_NAME = 'ct_GIST'
DATASET_ROOT = repo_root / 'gist'
prepared, paths = prepare_for_sam3d(
    dataset_root=DATASET_ROOT,
    sam3d_root=SAM3D_ROOT,
    category=CATEGORY,
    ct_name=CT_NAME,
    case_glob=None, max_cases=None,
)
print('Prepared cases:', prepared)
paths

## Step 3: Create folder-level validation split (for caching only)

In [ ]:
from med3pipe.data import split_validation
n_tr, n_val = split_validation(paths, split_ratio=0.8, seed=2025, copy=True)
print('Split done | Train:', n_tr, '| Val:', n_val)
paths.val_root

## Step 4: Build SAM-Med3D encoder and extract embeddings

In [ ]:
from med3pipe.sam.core import build_sam3d_model, default_feature_dirs, extract_embeddings_train_val
model = build_sam3d_model(sam3d_root=SAM3D_ROOT, model_type='vit_b_ori', checkpoint=None)
feat_dirs = default_feature_dirs(SAM3D_ROOT, category=paths.category, ct_name=paths.ct_name)
extract_embeddings_train_val(paths, model, sam3d_root=SAM3D_ROOT, img_size=128, feature_dirs=feat_dirs)
feat_dirs

## Step 5–6: ROI-pooled features and labels (map)

In [ ]:
from med3pipe.sam.core import load_labels_from_sheet
sheet_csv = repo_root / 'gist' / 'sheet.csv'
df, lab_map = load_labels_from_sheet(
    sheet_csv=sheet_csv, dataset_name='GIST', subject_col='Subject', label_col='Diagnosis_binary', case_suffix='_CT'
)
print('Labels loaded:', len(df))
df.head(3)

## Step 6b: Feature-level STRATIFIED split from union of features

In [ ]:
from med3pipe.tabular.stratify import stratified_features_split
(X_train, y_train, ids_train), (X_val, y_val, ids_val) = stratified_features_split(
    feat_train_dir=feat_dirs.train_dir,
    feat_val_dir=feat_dirs.val_dir,
    labels_tr_dir=paths.labels_tr,
    labels_val_dir=paths.labels_val,
    lab_map=lab_map,
    train_ratio=0.8, seed=2025,
)
X_train.shape, X_val.shape, len(ids_train), len(ids_val)

## Step 7: Standardize + PCA (fit on TRAIN only)

In [ ]:
from med3pipe.tabular import standardize_pca
X_train_p, X_val_p, scaler, pca = standardize_pca(X_train=X_train, X_val=X_val, n_components_max=128, random_state=42)
X_train_p.shape, X_val_p.shape

## Step 8: LoCalPFN (fast) — retrieval, optional adapter, inference

In [ ]:
import numpy as np, time
from med3pipe.tabular.localpfn import build_knn_index, retrieve_neighbors, _logit, _apply_adapter, _train_adapter, _auto_k
from med3pipe.tabular.tabpfn import ensure_tabpfn_on_sys_path
ensure_tabpfn_on_sys_path(None)
from tabpfn.classifier import TabPFNClassifier # type: ignore

K = 16
FIT_ADAPTER = False
ADAPTER_EPOCHS = 5
ADAPTER_LR = 5e-2
ADAPTER_WEIGHT_DECAY = 0.0
ADAPTER_NUM_QUERIES = 64
RETRIEVAL_METRIC = 'euclidean'
TABPFN_KW = {}

k_eff = int(K) if K is not None else int(_auto_k(X_train_p.shape[0]))
print('Effective k:', k_eff)
t0 = time.time(); knn = build_knn_index(X_train_p, metric=RETRIEVAL_METRIC); print('KNN fit:', f'{time.time()-t0:.3f}s')

adapter = None
if FIT_ADAPTER:
    print('[LoCalPFN-fast] Training adapter on local neighborhoods...')
    n_q = min(int(ADAPTER_NUM_QUERIES), X_train_p.shape[0])
    rng = np.random.default_rng(42)
    q_idx = rng.choice(X_train_p.shape[0], size=n_q, replace=False)
    logits_list, y_list = [], []
    clf = TabPFNClassifier(device=device_str, **TABPFN_KW)
    t_adapt = time.time()
    for qi in q_idx:
        idxs, _ = retrieve_neighbors(knn, X_train_p[qi:qi+1], k=k_eff)
        neigh = idxs[0]
        neigh = neigh[neigh != qi]
        if neigh.size == 0:
            continue
        X_ctx, y_ctx = X_train_p[neigh], y_train[neigh]
        clf.fit(X_ctx, y_ctx)
        proba = clf.predict_proba(X_train_p[qi:qi+1])
        if proba.shape[1] != 2:
            continue
        logit = _logit(proba[:, 1:2])
        logits_list.append(logit[0, 0])
        y_list.append(int(y_train[qi]))
    if len(logits_list) > 10:
        logits_arr = np.array(logits_list, dtype=np.float32)
        y_arr = np.array(y_list, dtype=np.float32)
        adapter = _train_adapter(logits=logits_arr, y=y_arr, epochs=ADAPTER_EPOCHS, lr=ADAPTER_LR, weight_decay=ADAPTER_WEIGHT_DECAY, verbose=True)
        print('Adapter trained in', f'{time.time()-t_adapt:.1f}s')
    else:
        print('[LoCalPFN-fast] Skipping adapter training (insufficient samples).')

# Inference on VAL
print('[LoCalPFN-fast] Running local-context inference on validation set...')
t_all = time.time()
idxs_val, _ = retrieve_neighbors(knn, X_val_p, k=k_eff)
clf = TabPFNClassifier(device=device_str, **TABPFN_KW)
y_pred = np.zeros((idxs_val.shape[0],), dtype=np.int64)
proba_buf = []
for i, neigh in enumerate(idxs_val):
    t_i = time.time()
    X_ctx, y_ctx = X_train_p[neigh], y_train[neigh]
    clf.fit(X_ctx, y_ctx)
    try:
        p = clf.predict_proba(X_val_p[i:i+1])
        if adapter is not None and p.shape[1] == 2:
            z = _logit(p[:, 1:2])
            z_adj = _apply_adapter(z, adapter)
            p1 = 1.0 / (1.0 + np.exp(-z_adj))
            p = np.concatenate([1 - p1, p1], axis=1)
        proba_buf.append(p[0])
    except Exception:
        pass
    y_pred[i] = int(clf.predict(X_val_p[i:i+1])[0])
    if (i + 1) % 5 == 0 or (i + 1) == len(idxs_val):
        print(f'[{i+1}/{len(idxs_val)}] iter {time.time()-t_i:.2f}s | total {time.time()-t_all:.1f}s')

print('VAL local inference took:', f'{time.time()-t_all:.1f}s')

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
acc = float(accuracy_score(y_val, y_pred))
macro_f1 = float(f1_score(y_val, y_pred, average='macro'))
roc_auc = None
if len(proba_buf) == len(y_pred):
    proba_val = np.stack(proba_buf, axis=0)
    if proba_val.shape[1] == 2:
        roc_auc = float(roc_auc_score(y_val, proba_val[:, 1]))
print('Accuracy:', acc)
print('Macro F1:', macro_f1)
print('ROC AUC:', roc_auc)
print(classification_report(y_val, y_pred, target_names=['0','1']))
confusion_matrix(y_val, y_pred)

# All at once: LoCalPFN across datasets from YAML (single-method)

In [ ]:
from med3pipe.pipelines import run_multi_localpfn
res_loc = run_multi_localpfn(
    config_path=config_path, outputs_base_dir=outputs_base, dataset_names=('gist',),
    local_k=8, local_fit_adapter=True, local_adapter_epochs=8, local_adapter_num_queries=150,
)
res_loc['summary_df']